# VoteMatrix - Anwendungsbeispiele

Dieser Notebook demonstriert die Verwendung der `VoteMatrix`-Klasse mit verschiedenen Konfigurationsoptionen:
- Einfaches Anwendungsbeispiel
- Zufällige vs. vorgegebene Parteipräferenzen
- Verschiedene Arten der Wahlbeteiligung
- Reproduzierbare Ergebnisse mit Seed
- Auswertung von Parteien und Stimmanteilen

In [ ]:
from ipres import VoteMatrix, Contestant, contestantsFromParties, Parties, ConstituenciesConfig
parties_config = Parties.from_csv("../../data/examples/de_parties.csv")
constituencies_config = ConstituenciesConfig.from_parquet("../../data/examples/de_bundestag_constituencies.parquet")

contestants : list[Contestant] = contestantsFromParties(parties_config.getPartyNames())
ballot = VoteMatrix.generate(constituencies_config, contestants)
figure = ballot.plot_vote_share_pie("Stimmenverteilung in %")

In [ ]:
from ipres.parties import Parties
from ipres import VoteMatrix, ConstituenciesConfig, Contestant, contestantsFromParties
import numpy as np
from IPython.display import display

# Kleine Beispielkonfiguration erstellen
M = 5
Smin, Smax = 10_000, 15_000
N = 4

const_cfg = ConstituenciesConfig.from_random(M=M, Smin=Smin, Smax=Smax, average_turnout_percent=70.0)
parties = Parties.from_random(N=N)
party_names = parties.getPartyNames()

display(const_cfg.getConstituencies().head())
display(parties.getParties())

## 1) Standard: zufällige Präferenzen und zufällige Wahlbeteiligung

- `probabilities=None`: wird intern zufällig erzeugt (Dirichlet)
- `turnout=None`: pro Wahlkreis zufällig (Beta-Verteilung mit moderater Streuung)

In [ ]:
vote_matrix_std = VoteMatrix.generate(
    constituencies_config=const_cfg,
    contestants=contestantsFromParties(party_names),
    probabilities=None,
    rng=None,
    turnout=None,
)

# Stimmenmatrix (Kopf) anzeigen
display(vote_matrix_std.getVotes().head())
# Tortendiagramm der Stimmenanteile
fig = vote_matrix_std.plot_vote_share_pie(title="Beispiel 1: Zufällige Präferenzen & Beteiligung")
#display(fig)

## 2) Vorgegebene Parteienwahrscheinlichkeiten (probabilities)

Die Wahrscheinlichkeiten werden als Mapping `Parteiname -> Prozentwert (0–100)` angegeben. Die Summe sollte 100 ergeben; falls nicht, normalisiert `VoteMatrix` automatisch auf 100%.

In [ ]:
party_names = parties.getPartyNames()
probs_map = {name: w for name, w in zip(party_names, [40.0, 30.0, 20.0, 10.0])}

vote_matrix_probs = VoteMatrix.generate(
    constituencies_config=const_cfg,
    contestants=contestantsFromParties(party_names),
    probabilities=probs_map,
    rng=np.random.default_rng(123),  # optional
    turnout=None,
)

display(vote_matrix_probs.getVotes().head())
fig = vote_matrix_probs.plot_vote_share_pie(title="Beispiel 2: Vorgegebene Wahrscheinlichkeiten")
display(fig)

## 2b) probabilities als Liste/Sequenz (in Prozent, Reihenfolge der Parteien)

Statt eines Mappings kann eine Sequenz (Liste/Tuple) von Prozentwerten in der Reihenfolge der `participating_parties` übergeben werden. Die Länge der Liste muss der Anzahl der Parteien entsprechen.

In [ ]:
probs_seq = [45.0, 30.0, 15.0, 10.0]  # in Prozent, Summe = 100
vote_matrix_probs_seq = VoteMatrix.generate(
    constituencies_config=const_cfg,
    contestants=contestantsFromParties(party_names),
    probabilities=probs_seq,
    rng=np.random.default_rng(321),
    turnout=None,
)

display(vote_matrix_probs_seq.getVotes().head())
fig = vote_matrix_probs_seq.plot_vote_share_pie(title="Beispiel 2b: Wahrscheinlichkeiten als Liste (in %)")
#display(fig)

## 3) Reproduzierbarer Zufall über rng (Seed)

Ein übergebenes `rng` (NumPy Generator) stellt reproduzierbare Ergebnisse sicher.

In [ ]:
rng_seeded = np.random.default_rng(42)
vote_matrix_seeded = VoteMatrix.generate(
    constituencies_config=const_cfg,
    contestants=contestantsFromParties(party_names),
    probabilities=None,   # zufällige Präferenzen, aber über rng reproduzierbar
    rng=rng_seeded,
    turnout=None,
)

display(vote_matrix_seeded.getVotes().head())
fig = vote_matrix_seeded.plot_vote_share_pie(title="Beispiel 3: Reproduzierbarer Zufall (rng=42)")
#display(fig)

## 4) Wahlbeteiligung als einzelner Prozentwert (Float)

`turnout=75.0` bedeutet: pro Wahlkreis werden Zufallswerte mit Mittelwert 75% (Beta-Verteilung) erzeugt.

In [ ]:
vote_matrix_turnout_float = VoteMatrix.generate(
    constituencies_config=const_cfg,
    contestants=contestantsFromParties(party_names),
    probabilities=None,
    rng=np.random.default_rng(7),
    turnout=75.0,  # Prozent
)

display(vote_matrix_turnout_float.getVotes().head())
fig = vote_matrix_turnout_float.plot_vote_share_pie(title="Beispiel 4: turnout=75% (als Float)")
display(fig)

## 5) Wahlbeteiligung je Wahlkreis als Mapping

`turnout` kann auch als Mapping `Wahlkreisname -> Prozent` (0–100) übergeben werden.

In [ ]:
const_names = const_cfg.getConstituencyNames()
# Beispiel: abwechselnd 60% und 80%
turnout_map = {name: (60.0 if i % 2 == 0 else 80.0) for i, name in enumerate(const_names)}

vote_matrix_turnout_map = VoteMatrix.generate(
    constituencies_config=const_cfg,
    contestants=contestantsFromParties(party_names),
    probabilities=None,
    rng=np.random.default_rng(9),
    turnout=turnout_map,
)

display(vote_matrix_turnout_map.getVotes().head())
fig = vote_matrix_turnout_map.plot_vote_share_pie(title="Beispiel 5: turnout als Mapping je Wahlkreis")
display(fig)

## 5b) Wahlbeteiligung als Liste/Sequenz (in Prozent)

Alternativ kann `turnout` als Liste/Sequenz von Prozentwerten für jeden Wahlkreis in der Reihenfolge von `constituencies_config.getConstituencyNames()` angegeben werden. Die Länge der Liste muss der Anzahl der Wahlkreise entsprechen.

In [ ]:
turnout_seq = [65.0 if i % 2 == 0 else 75.0 for i in range(len(const_names))]
vote_matrix_turnout_seq = VoteMatrix.generate(
    constituencies_config=const_cfg,
    contestants=contestantsFromParties(party_names),
    probabilities=None,
    rng=np.random.default_rng(11),
    turnout=turnout_seq,
)

display(vote_matrix_turnout_seq.getVotes().head())
fig = vote_matrix_turnout_seq.plot_vote_share_pie(title="Beispiel 5b: turnout als Liste (in %)")
display(fig)